# CC3092 – Deep Learning · Laboratorio 8
**Mini-GPT con muestreo controlado + embeddings contextuales (BERT) vs. estáticos (Word2Vec)**

Todo el Mini-GPT está construido *from scratch* con tensores y `nn.Parameter`/proyecciones lineales:
- Sin `nn.MultiheadAttention`, `nn.Transformer`, `nn.LayerNorm`.
- LayerNorm, softmax, cross-entropy, Positional Encoding (seno/coseno), atención multi-cabeza, PCA y similitud coseno programados a mano.
- Minibatches generados manualmente (índices), sin `DataLoader`.
- Learning rate con decaimiento expuesto en el bucle de entrenamiento + `clip_grad_norm_`.

Dependencias: `pip install torch numpy matplotlib pandas transformers gensim`

In [4]:
import os, math, time, re, itertools, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", device)

device: cpu


## Task 1.1 – Dataset, vocabulario y batches

In [5]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "input.txt"
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()
print("Caracteres totales:", len(text))

# Vocabulario de caracteres únicos y mappings bidireccionales char <-> índice
chars = sorted(set(text))
vocab_size = len(chars)
assert vocab_size == 65, f"Se esperaban 65 caracteres, hay {vocab_size}"
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

# Texto completo como secuencia de índices + split temporal 90/10 (sin barajar)
data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print("train:", len(train_data), "| val:", len(val_data), "| vocab:", vocab_size)

# ---------------- Hiperparámetros ----------------
block_size = 128
d_model    = 128
n_heads    = 4
n_layers   = 3
batch_size = 64
lr         = 3e-4
epochs     = 10
# Extras (no contradicen el enunciado)
dropout      = 0.1
d_ff         = 4 * d_model
grad_clip    = 1.0      # clip_grad_norm_ contra explosión de gradientes
warmup_steps = 100
lr_min       = 3e-5
STRIDE       = 64       # paso de la ventana deslizante en entrenamiento (menor = más pasos por época)

Caracteres totales: 1115394
train: 1003854 | val: 111540 | vocab: 65


In [6]:
# ---------------- Generación manual de batches (ventanas deslizantes por índices) ----------------
def make_window_starts(n_tokens, block_size, stride):
    # Un inicio s es válido si s + block_size + 1 <= n_tokens (necesitamos el target desplazado +1)
    last_start = n_tokens - block_size - 1
    return torch.arange(0, last_start + 1, stride)

def build_batch(split_data, starts, block_size):
    # idx[b, t] = starts[b] + t  ->  x = data[idx],  y = data[idx + 1]  (target = input desplazado 1 paso)
    offs = torch.arange(block_size)
    idx = starts[:, None] + offs[None, :]
    x = split_data[idx]
    y = split_data[idx + 1]
    return x.to(device), y.to(device)

def epoch_batches(split_data, block_size, batch_size, stride, shuffle=True, drop_last=True):
    starts = make_window_starts(len(split_data), block_size, stride)
    if shuffle:
        starts = starts[torch.randperm(len(starts))]    # permutación de índices
    n = len(starts)
    end = (n // batch_size) * batch_size if drop_last else n
    for i in range(0, end, batch_size):
        yield build_batch(split_data, starts[i:i + batch_size], block_size)

# Sanity check: y es x desplazado un paso
xb, yb = next(epoch_batches(train_data, block_size, batch_size, STRIDE))
assert torch.equal(xb[:, 1:], yb[:, :-1])
print("batch:", xb.shape, yb.shape)
print("x[0][:30] ->", repr(decode(xb[0, :30].cpu())))
print("y[0][:30] ->", repr(decode(yb[0, :30].cpu())))

batch: torch.Size([64, 128]) torch.Size([64, 128])
x[0][:30] -> 'e,\nA thirsty evil; and when we'
y[0][:30] -> ',\nA thirsty evil; and when we '
